In [1]:
import torch
from pathlib import Path
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import xml.etree.ElementTree as ET
from torch.utils.data import DataLoader
import torchvision.transforms as T
device = torch.device('mps')

In [2]:
folder_with_images = Path(r"/Users/igor.varha/CCP_workdir/skull_workdir/head_image_top")
folder_with_labels = Path(r"/Users/igor.varha/CCP_workdir/skull_workdir/head_label_top")


In [3]:

def read_classes_from_xmls(annotations_dir):
    """
    Reads all unique class names from Pascal VOC XML files.

    Args:
        annotations_dir (str): Path to the directory containing XML annotation files.

    Returns:
        List[str]: A list of unique class names.
    """
    classes = set()  # Use a set to avoid duplicates

    for xml_file in os.listdir(annotations_dir):
        if not xml_file.endswith(".xml"):
            continue

        xml_path = os.path.join(annotations_dir, xml_file)

        # Skip empty or invalid XML files
        if os.path.getsize(xml_path) == 0:
            print(f"Warning: Skipping empty file: {xml_file}")
            continue

        try:
            # Parse the XML file
            tree = ET.parse(xml_path)
            root = tree.getroot()

            # Extract class names
            for obj in root.findall("object"):
                class_name = obj.find("name").text
                classes.add(class_name)
        except ET.ParseError as e:
            print(f"Error parsing {xml_file}: {e}")
            continue

    return sorted(list(classes))  # Return as a sorted list # Return as a sorted list


class PascalVOCDataset(Dataset):
    def __init__(self, images_dir, annotations_dir, classes, transform=None):
        self.images_dir = images_dir
        self.annotations_dir = annotations_dir
        self.classes = classes
        self.transform = transform or T.ToTensor()  # Default transform

        # Filter image files to only include those with corresponding annotation files
        self.image_files = [
            f for f in os.listdir(images_dir)
            if f.endswith(".png") and os.path.exists(
                os.path.join(annotations_dir, os.path.splitext(f)[0] + ".xml")
            )
        ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.images_dir, image_file)
        annotation_file = os.path.join(self.annotations_dir, os.path.splitext(image_file)[0] + ".xml")

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Parse XML annotation
        boxes = []
        labels = []
        tree = ET.parse(annotation_file)
        root = tree.getroot()
        for obj in root.findall("object"):
            class_name = obj.find("name").text
            if class_name not in self.classes:
                continue
            labels.append(self.classes.index(class_name))
            bbox = obj.find("bndbox")
            xmin = round(float(bbox.find("xmin").text))
            ymin = round(float(bbox.find("ymin").text))
            xmax = round(float(bbox.find("xmax").text))
            ymax = round(float(bbox.find("ymax").text))
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transform:
            image = self.transform(image)

        return image, target

In [8]:
classes = read_classes_from_xmls("/Users/igor.varha/CCP_workdir/ref_mouse")
classes = ['__background__'] + classes  # Add background class
print(classes)

['__background__', 'eye1', 'eye2', 'rightb1', 'rightb2']


In [5]:

dataset = PascalVOCDataset(
    images_dir=folder_with_images,
    annotations_dir=folder_with_labels,
    classes=classes
)

# Split into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))



In [6]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, FasterRCNN_ResNet50_FPN_Weights

# Load Faster R-CNN with ResNet-50 backbone
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

# Adjust the model for your number of classes
num_classes = len(classes)  # Number of classes, including the background
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Replace the pre-trained head with a new one
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [7]:
import torch.optim as optim


# Define evaluation function
def  evaluate_model(model, val_loader, device):
    model.train()  # Set to training mode for loss computation
    val_loss = 0.0

    with torch.no_grad():  # Disable gradient computation for validation
        for images, targets in val_loader:
            # Move images and targets to the device
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Compute validation loss
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()

    # Calculate average loss over the validation set
    avg_val_loss = val_loss / len(val_loader)
    model.eval()  # Switch back to evaluation mode
    return avg_val_loss

    # Calculate average loss over the validation set
    avg_val_loss = val_loss / len(val_loader)
    return avg_val_loss

from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Initialize mAP metric
map_metric = MeanAveragePrecision()

def compute_map(model, val_loader, device):
    model.eval()
    map_metric.reset()

    with torch.no_grad():
        for images, targets in val_loader:
            # Move images to the correct device
            images = [img.to(device) for img in images]

            # Make predictions
            predictions = model(images)
            #print(predictions)
            # Convert targets to the format required by torchmetrics
            ground_truths = [
                {
                    "boxes": t["boxes"].to(device),  # Ensure on the same device
                    "labels": t["labels"].to(device),  # Ensure on the same device
                }
                for t in targets
            ]

            # Update metric
            map_metric.update(predictions, ground_truths)

    # Compute mAP
    map_results = map_metric.compute()
    return map_results

# Optimiser
optimizer = optim.SGD(model.parameters(), lr=0.0005, momentum=0.9, weight_decay=0.0005)

# Training Loop
device = torch.device('mps')
#print(torch.mps.device_count())
#torch.mps.empty_cache()
device = torch.device('cpu')
model.to(device)
num_epochs = 25
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    train_loss = 0.0

    for images, targets in train_loader:
        # print(1)
        # Move images and targets to the device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass and compute loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass and optimisation
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

        optimizer.step()

        train_loss += losses.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    avg_val_loss = evaluate_model(model, val_loader, device)
    map_results = compute_map(model, val_loader, device)

    # Log Results
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")
    print(f"mAP: {map_results['map']:.4f}, mAP@50: {map_results['map_50']:.4f}")

ModuleNotFoundError: No module named 'torchmetrics'

In [40]:
# find images with no labels and label them with trained model

def predict_image(model, image, device, threshold, num_classes):
    """
    Predicts bounding boxes and labels for an image using a trained model.

    Args:
        model: Trained object detection model.
        image: PIL Image to be predicted.
        device: Device to perform inference (e.g., "cpu" or "cuda").
        threshold: Confidence threshold for filtering predictions.
        num_classes: Total number of object classes.

    Returns:
        Tuple of filtered bounding boxes and labels.
    """
    # Convert image to tensor
    image_tensor = T.ToTensor()(image).unsqueeze(0).to(device)

    # Make prediction
    model.eval()
    with torch.no_grad():
        prediction = model(image_tensor)

    # Get boxes, scores, and labels
    boxes = prediction[0]["boxes"]
    scores = prediction[0]["scores"]
    labels = prediction[0]["labels"]

    # Filter by confidence threshold
    valid_indices = scores > threshold
    boxes = boxes[valid_indices]
    scores = scores[valid_indices]
    labels = labels[valid_indices]

    # Enforce exactly one detection per class
    final_boxes = []
    final_labels = []

    for cls in range(1, num_classes + 1):  # Skip background (class 0)
        cls_indices = labels == cls
        if cls_indices.sum() > 0:
            cls_boxes = boxes[cls_indices]
            cls_scores = scores[cls_indices]
            # Select the box with the highest confidence
            best_idx = torch.argmax(cls_scores)
            final_boxes.append(cls_boxes[best_idx].tolist())
            final_labels.append(cls)

    return final_boxes, final_labels


def convert_result_to_pascal_voc(boxes, labels, class_names, image_size, output_path, image_name,folder_name):
    """
    Converts predicted boxes and labels to Pascal VOC XML format.

    Args:
        boxes (list or numpy.ndarray): List of bounding boxes in the format [xmin, ymin, xmax, ymax].
        labels (list or numpy.ndarray): List of corresponding class labels for the boxes.
        class_names (list): List of class names where index corresponds to the label.
        image_size (tuple): Image size as (width, height, depth).
        output_path (str): Path to save the Pascal VOC XML file.
        image_name (str): Name of the image (without the extension).

    Returns:
        None
    """
    # Create the root XML element
    annotation = ET.Element("annotation")

    # Add image metadata
    folder = ET.SubElement(annotation, "folder")
    folder.text = folder_name

    filename = ET.SubElement(annotation, "filename")
    filename.text = f"{image_name}"

    size = ET.SubElement(annotation, "size")
    width = ET.SubElement(size, "width")
    width.text = str(image_size[0])
    height = ET.SubElement(size, "height")
    height.text = str(image_size[1])
    depth = ET.SubElement(size, "depth")
    depth.text = "1"

    # Add each object
    for box, label in zip(boxes, labels):
        obj = ET.SubElement(annotation, "object")

        name = ET.SubElement(obj, "name")
        name.text = class_names[label]

        pose = ET.SubElement(obj, "pose")
        pose.text = "Unspecified"

        truncated = ET.SubElement(obj, "truncated")
        truncated.text = "0"

        difficult = ET.SubElement(obj, "difficult")
        difficult.text = "0"

        bndbox = ET.SubElement(obj, "bndbox")
        xmin = ET.SubElement(bndbox, "xmin")
        xmin.text = str(int(box[0]))
        ymin = ET.SubElement(bndbox, "ymin")
        ymin.text = str(int(box[1]))
        xmax = ET.SubElement(bndbox, "xmax")
        xmax.text = str(int(box[2]))
        ymax = ET.SubElement(bndbox, "ymax")
        ymax.text = str(int(box[3]))

    # Save the XML to a file
    tree = ET.ElementTree(annotation)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)


aut_label_dir = Path(r"/Users/igor.varha/CCP_workdir/full_body_workdir/automated_label")

# iterate over images

for image_file in os.listdir(folder_with_images):
    if (folder_with_labels/Path(image_file).with_suffix(".xml")).exists():
        print(f"Skipping {image_file}", (folder_with_labels/Path(image_file).with_suffix(".xml")))
        continue
    image = Image.open(folder_with_images/image_file).convert("RGB")
    boxes, labels = predict_image(model, image, device, threshold=0.2, num_classes=len(classes))
    convert_result_to_pascal_voc(boxes, labels, classes, image.size, aut_label_dir/Path(image_file).with_suffix(".xml"), image_file, aut_label_dir.name)


Skipping Col2a1_F_E64-CL643.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/Col2a1_F_E64-CL643.xml
Skipping wt_F_82Z-CD065.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/wt_F_82Z-CD065.xml
Skipping Fam129a_F_E64-F5946.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/Fam129a_F_E64-F5946.xml
Skipping SMIM24-2M-X_M_E64-F6074.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/SMIM24-2M-X_M_E64-F6074.xml
Skipping Col2a1_M_E64-CL617.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/Col2a1_M_E64-CL617.xml
Skipping Fam129a_F_E64-F5979.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/Fam129a_F_E64-F5979.xml
Skipping Fam129a_F_E64-F5978.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/Fam129a_F_E64-F5978.xml
Skipping SMIM24-2M-X_M_E64-F6066.png /Users/igor.varha/CCP_workdir/full_body_workdir/full_body_top_label/SMIM24-2M-X_M_E64-F6066.xm

In [41]:
# save model

torch.save(model.state_dict(), "/Users/igor.varha/CCP_workdir/full_body_workdir/top_model.pth")

In [38]:
list(range( len(classes)))

[0, 1, 2, 3, 4, 5]

In [39]:
len(classes)

6

In [46]:
targets[0]['boxes'].shape

torch.Size([5, 4])